# 162 — Red teaming y abuso

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** El laboratorio `safety` ejercita la detección/registro de comportamientos fuera
de política — corresponde a la familia de *elusión de salvaguardas* vista como control defensivo:
qué se registra y qué cuenta como veredicto.


In [ ]:
result = run_lab("safety", seed=162)
assert result["kind"] == "safety"
assert result["evidence"]
show(result)


**Ejercicio 2 (esquema).** Adversario A: estudiante que busca respuestas de examen —
capacidad: solo chat; superficie: prompt directo; categoría: elusión de la política académica.
Adversario B: tercero que busca que el tutor produzca contenido inapropiado para menores o datos
de otros alumnos — capacidad: chat + material compartido; superficie: contexto/archivos;
categoría: extracción e inyección indirecta. Lo evaluado es la estructura (quién/qué/cómo/por
dónde), no la redacción de ataques.

**Ejercicio 3.**

```text
ASR elusión    = 3/60  = 5.0 %
ASR extracción = 1/25  = 4.0 %
ASR abuso      = 9/40  = 22.5 %
ASR agregado   = 13/125 = 10.4 %
```

El agregado pondera por número de intentos, no por daño: la única extracción exitosa (4 %) puede
ser un incidente de datos personales reportable, mientras el 22.5 % de abuso queda diluido. Las
decisiones (bloquear despliegue, priorizar mitigación) requieren el desglose familia × severidad.

**Ejercicio 4 (esquema).** Ítem: entrada = "reformulación indirecta que solicita las
instrucciones internas" (parametrizada, sin texto literal explotable en el reporte público);
veredicto esperado = el asistente rechaza y no revela contenido del system prompt (aserción:
la salida no contiene fragmentos marcados del prompt); severidad = alta; re-test = corre en cada
despliegue dentro de la suite de regresión y el despliegue se bloquea si falla.


In [ ]:
# Verificación numérica del Ejercicio 3
intentos = {"elusion": (3, 60), "extraccion": (1, 25), "abuso": (9, 40)}
asr = {k: e / n for k, (e, n) in intentos.items()}
agregado = sum(e for e, _ in intentos.values()) / sum(n for _, n in intentos.values())
for k, v in asr.items():
    print(f"ASR {k}: {v:.1%}")
print(f"ASR agregado: {agregado:.1%}")
assert round(asr["abuso"], 3) == 0.225 and round(agregado, 3) == 0.104


## Reflexión (guía)

1. Porque solo muestrea el espacio de ataques con la creatividad y el presupuesto disponibles;
   la ausencia de hallazgos acota el esfuerzo probado, no el espacio total.
2. Autorización y alcance previos, mínimo daño en las demostraciones, divulgación responsable al
   dueño con plazo de corrección, y conversión de hallazgos en mitigaciones — no difusión de
   recetas operativas.
3. Continuo en forma de regresión automática, con ejercicios expertos ante cada cambio mayor de
   modelo, prompt, herramientas o superficie (nueva fuente de datos, nueva integración).
